In [1]:
import sys
import os
import psycopg2
from dotenv import load_dotenv
import QuantLib as ql


sys.path.append(os.path.abspath(".."))
from utils.black_scholes import get_implied_vol

load_dotenv()

url = os.getenv("DATABASE_URL")

In [2]:
conn = psycopg2.connect(url, sslmode="require")
cur = conn.cursor()

In [3]:
postfix_list = [
    "CP6D", #04-22
]
expiry_date = ql.Date(22, 4, 2026)
eval_date = ql.Date(16, 4, 2026)

ticker = "SR"
min_strike = 270
max_strike = 370
strike_step = 10

strike = min_strike
tickers = []
while strike < max_strike:
    for postfix in postfix_list:
        option_ticker = ticker + str(strike) + postfix
        tickers.append(option_ticker)
    strike += strike_step
print(tickers)

['SR270CP6D', 'SR280CP6D', 'SR290CP6D', 'SR300CP6D', 'SR310CP6D', 'SR320CP6D', 'SR330CP6D', 'SR340CP6D', 'SR350CP6D', 'SR360CP6D']


In [4]:
query = """
SELECT DISTINCT ON (ticker)
    ticker,
    bids,
    asks
FROM orderbooks
WHERE ticker = ANY(%s)
ORDER BY ticker, timestamp DESC;
"""

cur.execute(query, (tickers,))
rows = cur.fetchall()

In [5]:
rows

[('SR270CP6D', [], [{'price': 0.55, 'quantity': 1}]),
 ('SR280CP6D', [], [{'price': 0.55, 'quantity': 1}]),
 ('SR290CP6D',
  [],
  [{'price': 0.06, 'quantity': 800}, {'price': 0.57, 'quantity': 800}]),
 ('SR300CP6D',
  [{'price': 0.04, 'quantity': 63871}],
  [{'price': 0.06, 'quantity': 800},
   {'price': 0.6, 'quantity': 800},
   {'price': 1.98, 'quantity': 800}]),
 ('SR310CP6D',
  [{'price': 0.08, 'quantity': 25000},
   {'price': 0.03, 'quantity': 30000},
   {'price': 0.02, 'quantity': 100000},
   {'price': 0.01, 'quantity': 800}],
  [{'price': 0.16, 'quantity': 91000},
   {'price': 0.18, 'quantity': 800},
   {'price': 0.4, 'quantity': 10},
   {'price': 0.78, 'quantity': 800},
   {'price': 0.9, 'quantity': 800}]),
 ('SR320CP6D',
  [{'price': 1.0, 'quantity': 10650},
   {'price': 0.97, 'quantity': 800},
   {'price': 0.78, 'quantity': 15},
   {'price': 0.72, 'quantity': 800},
   {'price': 0.71, 'quantity': 3000}],
  [{'price': 1.78, 'quantity': 5500},
   {'price': 1.79, 'quantity': 800

In [6]:
def compute_mid_price(bids, asks):
    best_bid = None
    best_ask = None
    
    if bids:
        best_bid = max(bids, key=lambda x: x["price"])["price"]
    if asks:
        best_ask = min(asks, key=lambda x: x["price"])["price"]

    if best_bid is not None and best_ask is not None:
        return (best_bid + best_ask) / 2
    return best_bid or best_ask

In [7]:
result = []

for ticker, bids, asks in rows:
    mid = compute_mid_price(bids, asks)

    result.append({
        "ticker": ticker,
        "mid": mid,
    })
result

[{'ticker': 'SR270CP6D', 'mid': 0.55},
 {'ticker': 'SR280CP6D', 'mid': 0.55},
 {'ticker': 'SR290CP6D', 'mid': 0.06},
 {'ticker': 'SR300CP6D', 'mid': 0.05},
 {'ticker': 'SR310CP6D', 'mid': 0.12},
 {'ticker': 'SR320CP6D', 'mid': 1.3900000000000001},
 {'ticker': 'SR330CP6D', 'mid': 7.395},
 {'ticker': 'SR340CP6D', 'mid': 16.85},
 {'ticker': 'SR350CP6D', 'mid': 26.89},
 {'ticker': 'SR360CP6D', 'mid': 42.8}]

In [8]:
import re

def parse_ticker(ticker):
    m = re.match(r"SR(\d+)(CP|PP)\d+D", ticker)
    if not m:
        return None, None

    strike = float(m.group(1))
    option_type = "call" if m.group(2) == "CP" else "put"

    return strike, option_type

In [11]:
data = result
spot_price = 322
risk_free_rate = 0.15

result = []

for r in data:
    ticker = r["ticker"]
    mid = r["mid"]

    strike, option_type = parse_ticker(ticker)
    if strike is None:
        continue

    try:
        iv = get_implied_vol(
            market_price=mid,
            spot_price=spot_price,
            strike_price=strike,
            risk_free_rate=risk_free_rate,
            expiry_date=expiry_date,
            eval_date=eval_date,
            option_type=option_type
            )
    except Exception as e:
        iv = None
        print(e)
        

    result.append({
        "strike": strike,
        "mid": mid,
        "iv": iv,
        "ticker": ticker
    })
result

root not bracketed: f[1e-06,10] -> [5.211493e+01,1.684532e+02]
root not bracketed: f[1e-06,10] -> [4.213956e+01,1.654314e+02]
root not bracketed: f[1e-06,10] -> [3.265419e+01,1.629960e+02]
root not bracketed: f[1e-06,10] -> [2.268881e+01,1.601722e+02]
root not bracketed: f[1e-06,10] -> [1.264344e+01,1.573558e+02]
root not bracketed: f[1e-06,10] -> [1.398069e+00,1.534224e+02]


[{'strike': 270.0, 'mid': 0.55, 'iv': None, 'ticker': 'SR270CP6D'},
 {'strike': 280.0, 'mid': 0.55, 'iv': None, 'ticker': 'SR280CP6D'},
 {'strike': 290.0, 'mid': 0.06, 'iv': None, 'ticker': 'SR290CP6D'},
 {'strike': 300.0, 'mid': 0.05, 'iv': None, 'ticker': 'SR300CP6D'},
 {'strike': 310.0, 'mid': 0.12, 'iv': None, 'ticker': 'SR310CP6D'},
 {'strike': 320.0,
  'mid': 1.3900000000000001,
  'iv': None,
  'ticker': 'SR320CP6D'},
 {'strike': 330.0,
  'mid': 7.395,
  'iv': 0.6368902230378441,
  'ticker': 'SR330CP6D'},
 {'strike': 340.0,
  'mid': 16.85,
  'iv': 1.4501628019813884,
  'ticker': 'SR340CP6D'},
 {'strike': 350.0,
  'mid': 26.89,
  'iv': 2.279441181588604,
  'ticker': 'SR350CP6D'},
 {'strike': 360.0,
  'mid': 42.8,
  'iv': 3.448454809086917,
  'ticker': 'SR360CP6D'}]